# Qwen3 8B Dev Baseline

Closed-book baseline evaluation for `Qwen/Qwen3-8B` on the dev preview benchmark.

## Setup

Run in Colab with a GPU runtime. No Hugging Face token should be needed for this public model.

In [ ]:
!pip install -q -U transformers accelerate pandas tqdm

In [ ]:
from pathlib import Path
import json
import sys

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
PROJECT_ROOT = Path("/content/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
from training_eval.eval_utils import (
    default_dev_dir,
    extract_json_object,
    is_correct,
    load_jsonl_records,
    make_closed_book_prompt,
    rows_to_frame,
    save_results,
    summarize_accuracy,
)

## Load Data

In [ ]:
DATA_DIR = default_dev_dir(PROJECT_ROOT)
records = load_jsonl_records(DATA_DIR)
len(records), DATA_DIR

## Load Model

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Generate

In [ ]:
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0

def generate_answer(problem):
    messages = [{"role": "user", "content": make_closed_book_prompt(problem)}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
rows = []

for record in tqdm(records):
    raw_output = generate_answer(record["problem"])
    predicted = extract_json_object(raw_output)
    metadata = record.get("metadata", {})

    rows.append({
        "id": record["id"],
        "family": record["family"],
        "problem_type": record["problem_type"],
        "difficulty": record["difficulty"],
        "manual_variation": metadata.get("manual_variation", False),
        "manual_problem_variation": metadata.get("manual_problem_variation", False),
        "manual_reasoning_variation": metadata.get("manual_reasoning_variation", False),
        "problem": record["problem"],
        "canonical_answer": record["canonical_answer"],
        "raw_output": raw_output,
        "predicted_answer": predicted,
        "correct": is_correct(predicted, record["canonical_answer"]),
    })

df = rows_to_frame(rows)
df.head()

## Metrics

In [ ]:
print(f"Overall accuracy: {df['correct'].mean():.3f} ({df['correct'].sum()}/{len(df)})")
display(df.groupby("family")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("difficulty")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("manual_variation")["correct"].agg(["mean", "sum", "count"]).sort_index())

## Save Results

In [ ]:
result_dir = PROJECT_ROOT / "results" / "baselines" / "qwen3_8b_dev_closed_book"

metrics = summarize_accuracy(df)
metrics.update({
    "model_name": MODEL_NAME,
    "prompt_mode": "closed_book",
    "dataset_dir": str(DATA_DIR.relative_to(PROJECT_ROOT)),
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
})

save_results(rows, result_dir, metrics)